# Experiment Comparison

Read-only comparison of the newest evaluated run of every version. Optional paired source-group bootstrap CIs compare each version with v0 on identical development images.

**Inputs:** notebooks/experiments/v*/v*.yaml (names); runs/v*-YYYY-MM-DD/{eval/metrics.json,eval/paired_stats.npz,train/results.csv,data/metadata.csv}.

**Outputs:** In-memory tables and figures only.

**Prerequisites:** Completed 04 for each version to include. No training, inference or pixel access.

**Outline**

- [1. Configuration](#section-1)
- [2. Development Results](#section-2)
- [3. Paired Comparison Summary](#section-3)

<a id="section-1"></a>

## 1. Configuration

Pin a run per version with RUN_DATES, e.g. {"v1": "2026-11-09"}. BOOTSTRAP_RESAMPLES=0 skips the CPU-heavy CI.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from vehicle_detection.common import require_paths
from vehicle_detection.data import FIVE_CLASSES
from vehicle_detection.evaluation import saved_result
from vehicle_detection.evaluation_stats import load_stats, paired_bootstrap
from vehicle_detection.experiment_data import experiments, load_experiment

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
EXPERIMENTS = {v: load_experiment(ROOT, v)["name"] for v in experiments(ROOT)}
RUN_DATES = {}
BOOTSTRAP_RESAMPLES = 0
require_paths(runs=ROOT / "runs")
results = {}
for version in EXPERIMENTS:
    found = saved_result(ROOT, version, RUN_DATES.get(version))
    if found:
        results[version] = found
    else:
        print(f"{version}: no evaluated run yet")

<a id="section-2"></a>

## 2. Development Results

Same 13,975 development images/GT for every version; each detector sees its own policy.

In [ ]:
overall = pd.DataFrame(
    {
        v: {
            "run": run.name,
            **{k: m["overall"][k] * 100 for k in ("mAP50", "mAP50_95", "precision", "recall")},
            "dark_mAP50_95": m["cohorts"]["dark"]["mAP50_95"] * 100,
            "bright_mAP50_95": m["cohorts"]["bright"]["mAP50_95"] * 100,
        }
        for v, (run, m) in results.items()
    }
).T
display(overall)
classes = pd.DataFrame(
    {
        v: {row["Class"]: row["mAP50-95"] * 100 for row in m["classes"]}
        for v, (_, m) in results.items()
    }
).reindex(FIVE_CLASSES)
display(classes.round(3))
if not classes.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), layout="constrained")
    classes.plot.bar(ax=axes[0], title="Development AP50:95 by class (%)", rot=0)
    for v, (run, _) in results.items():
        history = pd.read_csv(run / "train/results.csv")
        history.columns = history.columns.str.strip()
        axes[1].plot(
            history.epoch, history["metrics/mAP50-95(B)"] * 100, label=f"{v} · {EXPERIMENTS[v]}"
        )
    axes[1].set(xlabel="Epoch", ylabel="%", title="Calibration selection curve")
    axes[1].legend(fontsize=8)
    plt.show()

<a id="section-3"></a>

## 3. Paired Comparison Summary

Group CIs are conditional on the trained single-seed models; no multiplicity correction.

In [ ]:
if BOOTSTRAP_RESAMPLES and "v0" in results:
    base_run, _ = results["v0"]
    metadata = pd.read_csv(base_run / "data/metadata.csv", keep_default_na=False)
    groups = metadata[metadata.split == "test"].set_index("id").source_group.to_dict()
    before = load_stats(base_run / "eval/paired_stats.npz")
    rows = {}
    for v, (run, _) in results.items():
        if v == "v0":
            continue
        change = paired_bootstrap(
            before,
            load_stats(run / "eval/paired_stats.npz"),
            groups,
            range(len(FIVE_CLASSES)),
            resamples=BOOTSTRAP_RESAMPLES,
        )
        rows[v] = {
            "delta_pp": change["delta"] * 100,
            "ci_low_pp": change["ci95"][0] * 100,
            "ci_high_pp": change["ci95"][1] * 100,
            "valid_draws": change["valid_draws"],
        }
    display(pd.DataFrame(rows).T)
else:
    print("Set BOOTSTRAP_RESAMPLES (e.g. 200) for paired source-group CIs against v0.")
print("Development only; final holdout stays sealed.")